In [ ]:
import os
import pandas as pd
import xgboost
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, train_test_split, GridSearchCV
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
import warnings

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [ ]:
def train_xgb(df, y_pred_baseline):
  df["datetime_idx"] = pd.to_datetime(df["datetime_idx"].astype(str))

  valid_test_dates = [str(date) for date in set(df["datetime_idx"]).intersection(set(y_pred_baseline["datetime_idx"]))]

  df.set_index("datetime_idx", inplace=True)
  y_pred_baseline.set_index("datetime_idx", inplace=True)

  X = df[[
      'minutes_from_open', 'minutes_to_close',
      'wait_time_lag_day', 'wait_time_lag_week', 'wait_time_lag_year',
      'wt_rolling_week_avg', 'wt_rolling_month_avg', 'wt_rolling_year_avg',
      'wt_rolling_week_max', 'wt_rolling_month_max', 'wt_rolling_year_max',
      'wt_rolling_week_std', 'wt_rolling_month_std', 'wt_rolling_year_std',
      'duration', 'min_height', 'has_express', "has_ea",
      'is_kids_ride', 'is_thrill_ride', 'is_dark_ride', 'is_water_ride', 'has_indoor_queue',
      'temp', 'rhum', 'prcp', 'wspd', 'pres', 'cldc', 'coco',
      'is_holiday', 'is_major_holiday', 'is_annual_event',
      'year', 'month_sin', 'month_cos', 'day_sin', 'day_cos', 'time_sin', 'time_cos',
      'day_of_week_sin', 'day_of_week_cos'
  ]]
  y = df["wait_time"]

  train_split = str(df[df.index < pd.to_datetime("2025-01-01 00:0:00")].index.max())

  X_train = X.loc[:train_split]
  X_test = X.loc[train_split:]
  X_test = X_test.loc[valid_test_dates].sort_index()

  y_train = y.loc[:train_split]
  y_test = y.loc[train_split:]
  y_test = y_test.loc[valid_test_dates].sort_index()
  y_pred_baseline = y_pred_baseline.loc[valid_test_dates].sort_index()

  X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)

  xgb = xgboost.XGBRegressor(
        objective='reg:tweedie',
        tweedie_variance_power=1.5,
        max_depth=6,
        learning_rate=0.05,
        n_estimators=500,
        early_stopping_rounds=50,
        subsample=0.6,
        random_state=42,
    )

  xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

  y_pred = pd.DataFrame(xgb.predict(X_test)).set_index(y_test.index)

  baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
  baseline_rmse = root_mean_squared_error(y_test, y_pred_baseline)
  baseline_wape = sum(np.abs(np.array(y_test) - np.array(y_pred_baseline["wait_time"]))) / sum(y_test)

  mae = mean_absolute_error(y_test, y_pred)
  rmse = root_mean_squared_error(y_test, y_pred)
  wape = sum(np.abs(np.array(y_test) - np.array(y_pred[0]))) / sum(y_test)

  return xgb, baseline_mae, baseline_rmse, baseline_wape, mae, rmse, wape, y_pred

def theme_park_xgb(folder_path):
  for file_name in os.listdir(f"{folder_path}Wait_Time_Features/"):
    ride_name = file_name.replace("_features.parquet", "")
    try:
      baseline_predictions = pd.read_parquet(f"{folder_path}Baseline_Predictions/{ride_name}.parquet")
      features = pd.read_parquet(f"{folder_path}Wait_Time_Features/{file_name}")
      #results = pd.read_csv(f"{folder_path}Results/xgb_results.csv")

      xgb, baseline_mae, baseline_rmse, baseline_wape, mae, rmse, wape, predictions = train_xgb(features, baseline_predictions)

      xgb.save_model(f"{folder_path}Models/{ride_name}_xgb.ubj")

      #results = pd.concat([results, pd.DataFrame({"ride_name": ride_name, "baseline_mae": baseline_mae, "baseline_rmse": baseline_rmse, "baseline_wape": baseline_wape, "mae": mae, "rmse": rmse, "wape": wape}, index=[0])], ignore_index=True)

      #results.to_csv(f"{folder_path}Results/xgb_results.csv", index=False)
      predictions.to_csv(f"{folder_path}Predictions/XGBoost/{ride_name}_predictions.csv", index=False)
    except Exception as e:
      continue


In [ ]:
# Create Linear Models for Islands of Adventure
theme_park_xgb("/content/drive/MyDrive/Theme_Park_Data/Universal_IOA/")

# Create Linear Models for Universal Studios
theme_park_xgb("/content/drive/MyDrive/Theme_Park_Data/Universal_Studios_FL/")